In [9]:
from ingest import load_faq_data
import time
from sqlitesearch import TextSearchIndex
from helpers import RAGBase
from openai import OpenAI

In [10]:
documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

Loaded 1342 documents


In [11]:
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

LLM Zoomcamp: 79 documents


In [12]:
index = TextSearchIndex(
	text_fields=["question", "section", "answer"],
	keyword_fields=["course"],
	db_path="faq.db"
)

for doc in docs_llm:
	index.add(doc)
	print(f"""Added: {doc["question"][:60]}...""")
	time.sleep(0.5)

index.close()
print("Done. Index saved to faq.db")

Added: I just discovered the course. Can I still join?...
Added: Course: I have registered for the LLM Zoomcamp. When can I e...
Added: What is the video/zoom link to the stream for the “Office Ho...
Added: Cloud alternatives with GPU...
Added: Leaderboard: I am not on the leaderboard / how do I know whi...
Added: Certificate: Can I follow the course in a self-paced mode an...
Added: I missed the first homework - can I still get a certificate?...
Added: Homework: Why does the content keep changing?...
Added: When will the course be offered next?...
Added: Are there any lectures/videos? Where are they?...
Added: WSL2: ResponseError: model requires more system memory (X.X ...
Added: Server Error (500) When logging in to course homework using ...
Added: Why are we not using Langchain in the course?...
Added: OpenAI: Error when running OpenAI responses.create command...
Added: OpenAI: Error: RateLimitError: Error code: 429 -...
Added: OpenAI: Error: 'Cannot import name OpenAI from openai';

In [13]:
sqlite_index = TextSearchIndex(
	text_fields=["question", "section", "answer"],
	keyword_fields=["course"],
	db_path="faq.db"
)

In [14]:
sqlite_index.count()

158

In [15]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'I missed the first homework - can I still get a certificate?',
 'Do we submit 2 projects, what does attempt 1 and 2 mean?']

In [16]:
openai_client = OpenAI()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.